# 3팀 공통 전처리

팀장 통합 CSV 기준으로 4개 공통 제거 기준을 적용한다. 팀원 전원이 동일 결과를 얻기 위한 공통 스크립트.

## 입력
- `merged_train_after_category_merge.csv`
- `final_mapping_after_category_merge.csv`

## 공통 제거 기준 (4개)
1. 명백한 오류 bbox: `w≤0`, `h≤0`, `area≤0`, `area NaN`
2. bbox 경계 초과: bbox가 이미지 범위(976×1280) 밖
3. 좌표 완전 중복: 같은 이미지 내 `bbox_x, bbox_y, bbox_w, bbox_h` 모두 동일
4. 라벨 누락/오류: 수민 로직(set 비교) + 같은 클래스 중복 라벨

## 점검 후보 (제거 X, 기록만)
- IoU < 1.0 near-duplicate
- area 오차율 > 10% / > 100% / > 500%
- 종횡비 극단, 짧은 변 20px 미만, area 상·하위 1%

## 제거 단위
이미지 통째. 문제 annotation이 하나라도 있으면 해당 이미지 전체 제외.

## 환경 설정

본인 환경에 맞게 경로만 수정.

In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

# 경로 (본인 환경에 맞게)
CSV_PATH     = r"D:\pill_detection\data\merged_train_after_category_merge.csv"
MAPPING_PATH = r"D:\pill_detection\data\final_mapping_after_category_merge.csv"
OUTPUT_DIR   = r"D:\pill_detection\data\team_preprocessed"

# 상수
IMG_WIDTH  = 976
IMG_HEIGHT = 1280

# 출력 폴더
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR, "review_candidates").mkdir(parents=True, exist_ok=True)

print(f"입력 CSV  : {CSV_PATH}")
print(f"매핑 CSV  : {MAPPING_PATH}")
print(f"출력 폴더 : {OUTPUT_DIR}")

입력 CSV  : D:\pill_detection\data\merged_train_after_category_merge.csv
매핑 CSV  : D:\pill_detection\data\final_mapping_after_category_merge.csv
출력 폴더 : D:\pill_detection\data\team_preprocessed


## Step 0. 데이터 로드

예상 수치: annotation 40,844 / 이미지 10,697 / 클래스 118

In [2]:
df = pd.read_csv(CSV_PATH, low_memory=False)
mapping = pd.read_csv(MAPPING_PATH)

print(f"annotation : {len(df):,}")
print(f"고유 이미지: {df['file_name'].nunique():,}")
print(f"클래스     : {df['category_id_merged'].nunique()}")
print(f"매핑 rows  : {len(mapping)}")

assert len(df) == 40844, f"annotation 수 불일치: {len(df)}"
assert df["file_name"].nunique() == 10697
assert df["category_id_merged"].nunique() == 118
print("\n데이터 검증 통과")

# 제거 대상 이미지 (이유별)
remove_images = {
    "invalid_bbox": set(),
    "out_of_bounds": set(),
    "duplicate_bbox": set(),
    "label_mismatch": set(),
}

annotation : 40,844
고유 이미지: 10,697
클래스     : 118
매핑 rows  : 118

데이터 검증 통과


## Step 1. 명백한 오류 bbox

`bbox_w ≤ 0`, `bbox_h ≤ 0`, `area ≤ 0`, `area NaN` 중 하나라도 해당하면 그 이미지 전체 제거.

In [3]:
invalid_mask = (
    (df["bbox_w"] <= 0) |
    (df["bbox_h"] <= 0) |
    (df["area"] <= 0) |
    (df["area"].isna())
)

invalid_df = df[invalid_mask]
invalid_imgs = set(invalid_df["file_name"].unique())
remove_images["invalid_bbox"] = invalid_imgs

print(f"명백한 오류 annotation: {invalid_mask.sum()}건")
print(f"  bbox_w <= 0 : {(df['bbox_w'] <= 0).sum()}")
print(f"  bbox_h <= 0 : {(df['bbox_h'] <= 0).sum()}")
print(f"  area <= 0   : {(df['area'] <= 0).sum()}")
print(f"  area NaN    : {df['area'].isna().sum()}")
print(f"제거 대상 이미지: {len(invalid_imgs)}장")

명백한 오류 annotation: 2건
  bbox_w <= 0 : 0
  bbox_h <= 0 : 0
  area <= 0   : 0
  area NaN    : 2
제거 대상 이미지: 2장


## Step 2. bbox 경계 초과

bbox가 이미지 범위 밖으로 나가는 경우:
- `bbox_x < 0` (왼쪽 밖)
- `bbox_y < 0` (위쪽 밖)
- `bbox_x + bbox_w > 976` (오른쪽 밖)
- `bbox_y + bbox_h > 1280` (아래쪽 밖)

In [4]:
oob_mask = (
    (df["bbox_x"] < 0) |
    (df["bbox_y"] < 0) |
    (df["bbox_x"] + df["bbox_w"] > IMG_WIDTH) |
    (df["bbox_y"] + df["bbox_h"] > IMG_HEIGHT)
)

oob_df = df[oob_mask]
oob_imgs = set(oob_df["file_name"].unique())
remove_images["out_of_bounds"] = oob_imgs

print(f"경계 초과 annotation: {oob_mask.sum()}건")
print(f"  bbox_x < 0           : {(df['bbox_x'] < 0).sum()}")
print(f"  bbox_y < 0           : {(df['bbox_y'] < 0).sum()}")
print(f"  bbox_x + w > 976     : {(df['bbox_x'] + df['bbox_w'] > IMG_WIDTH).sum()}")
print(f"  bbox_y + h > 1280    : {(df['bbox_y'] + df['bbox_h'] > IMG_HEIGHT).sum()}")
print(f"제거 대상 이미지: {len(oob_imgs)}장")

print("\nsource 분포:")
print(oob_df["source"].value_counts(dropna=False))

경계 초과 annotation: 1건
  bbox_x < 0           : 0
  bbox_y < 0           : 0
  bbox_x + w > 976     : 1
  bbox_y + h > 1280    : 0
제거 대상 이미지: 1장

source 분포:
source
NaN    1
Name: count, dtype: int64


## Step 3. 좌표 완전 중복

같은 이미지 내에서 `bbox_x, bbox_y, bbox_w, bbox_h` 4개 좌표가 모두 동일한 annotation. IoU=1.0.

In [5]:
dup_key = ["file_name", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]
dup_mask = df.duplicated(subset=dup_key, keep=False)

dup_df = df[dup_mask]
dup_imgs = set(dup_df["file_name"].unique())
remove_images["duplicate_bbox"] = dup_imgs

n_pairs = dup_mask.sum() // 2

print(f"좌표 완전 중복 annotation: {dup_mask.sum()}건 ({n_pairs}쌍)")
print(f"제거 대상 이미지: {len(dup_imgs)}장")

좌표 완전 중복 annotation: 142건 (71쌍)
제거 대상 이미지: 71장


## Step 4. 라벨 누락/오류

- **4-1. 수민님 로직** — 파일명 K-code → category_id 집합 vs 실제 category_id 집합 비교
- **4-2. 같은 클래스 중복 라벨** — set 비교가 놓치는 케이스 (같은 이미지에 같은 class 2번 이상)

In [6]:
# ----- 4-1. 수민님 로직 (set 비교) -----
mapping_dict = {}
for _, row in mapping.iterrows():
    if pd.notna(row["dl_mapping_code"]):
        code = int(row["dl_mapping_code"].replace("K-", ""))
        mapping_dict[code] = int(row["category_id_merged"])
    else:
        # NaN (Kaggle 전용 클래스)
        mapping_dict[int(row["category_id_merged"])] = int(row["category_id_merged"])

def parse_expected(filename):
    """파일명 K-code를 category_id 집합으로 변환"""
    prefix = filename.split("_0_")[0]
    codes = [int(x) for x in prefix.split("-")[1:]]
    return set(mapping_dict.get(c, c - 1) for c in codes)

annotated_cats = df.groupby("file_name")["category_id_merged"].apply(set).to_dict()

mismatch_files = set()
for fname, actual in annotated_cats.items():
    expected = parse_expected(fname)
    if (expected - actual) or (actual - expected):
        mismatch_files.add(fname)

print(f"[4-1] 수민 로직 mismatch: {len(mismatch_files)}장")

# ----- 4-2. 같은 클래스 중복 라벨 -----
class_dup = df.groupby(["file_name", "category_id_merged"]).size()
dup_class_files = set(class_dup[class_dup >= 2].reset_index()["file_name"].unique())
print(f"[4-2] 같은 클래스 중복 라벨: {len(dup_class_files)}장")

# union
label_bad = mismatch_files | dup_class_files
print(f"\n통합: {len(label_bad)}장")
print(f"  수민님 로직만 : {len(mismatch_files - dup_class_files)}장")
print(f"  중복만      : {len(dup_class_files - mismatch_files)}장")
print(f"  둘 다       : {len(mismatch_files & dup_class_files)}장")

remove_images["label_mismatch"] = label_bad

[4-1] 수민 로직 mismatch: 215장
[4-2] 같은 클래스 중복 라벨: 18장

통합: 232장
  수민님 로직만 : 214장
  중복만      : 17장
  둘 다       : 1장


## Step 5. 제거 대상 통합

4개 기준 union. 한 이미지가 여러 이유에 걸릴 수 있음.

In [7]:
all_remove = set()
for reason, imgs in remove_images.items():
    all_remove |= imgs

print("=" * 50)
print("제거 대상 이미지 (이유별, 중복 포함)")
print("=" * 50)
for reason, imgs in remove_images.items():
    print(f"  {reason:20s}: {len(imgs):>5}장")

print(f"\n최종 제거 이미지 (union): {len(all_remove):,}장")
print(f"잔존 이미지             : {df['file_name'].nunique() - len(all_remove):,}장")

# 이유별 중복 확인
img_reasons = {}
for img in all_remove:
    n = sum(1 for imgs in remove_images.values() if img in imgs)
    img_reasons[n] = img_reasons.get(n, 0) + 1

print("\n이미지별 중복 이유 분포:")
for n in sorted(img_reasons.keys()):
    print(f"  {n}개 이유에 걸림: {img_reasons[n]}장")

제거 대상 이미지 (이유별, 중복 포함)
  invalid_bbox        :     2장
  out_of_bounds       :     1장
  duplicate_bbox      :    71장
  label_mismatch      :   232장

최종 제거 이미지 (union): 305장
잔존 이미지             : 10,392장

이미지별 중복 이유 분포:
  1개 이유에 걸림: 304장
  2개 이유에 걸림: 1장


## Step 6. 점검 후보 기록

제거 X. 모델링/분석 시 참고용.

In [8]:
# area 오차율 계산
valid = df.dropna(subset=["area", "bbox_w", "bbox_h"]).copy()
valid = valid[(valid["bbox_w"] > 0) & (valid["bbox_h"] > 0) & (valid["area"] > 0)]
valid["bbox_area"] = valid["bbox_w"] * valid["bbox_h"]
valid["error_rate"] = (abs(valid["area"] - valid["bbox_area"]) / valid["bbox_area"]) * 100

rev_dir = Path(OUTPUT_DIR, "review_candidates")

# area 오차율 3단계
area_err_10  = valid[valid["error_rate"] > 10]
area_err_100 = valid[valid["error_rate"] > 100]
area_err_500 = valid[valid["error_rate"] > 500]

area_err_10.to_csv(rev_dir / "area_error_over_10.csv", index=False)
area_err_100.to_csv(rev_dir / "area_error_over_100.csv", index=False)
area_err_500.to_csv(rev_dir / "area_error_over_500.csv", index=False)

print("[area 오차율]")
print(f"  > 10%  (점검 후보)        : {len(area_err_10)}건")
print(f"  > 100% (오류 후보)        : {len(area_err_100)}건")
print(f"  > 500% (강한 오류 후보)   : {len(area_err_500)}건")

# 종횡비 극단
valid["aspect_ratio"] = valid["bbox_w"] / valid["bbox_h"]
ar_low  = valid["aspect_ratio"].quantile(0.01)
ar_high = valid["aspect_ratio"].quantile(0.99)
aspect_extreme = valid[(valid["aspect_ratio"] < ar_low) | (valid["aspect_ratio"] > ar_high)]
aspect_extreme.to_csv(rev_dir / "aspect_ratio_extreme.csv", index=False)
print(f"\n[종횡비] 하위 1% < {ar_low:.3f}, 상위 1% > {ar_high:.3f}: {len(aspect_extreme)}건")

# 짧은 변 20px 미만
short_side = np.minimum(valid["bbox_w"], valid["bbox_h"])
short_small = valid[short_side < 20]
short_small.to_csv(rev_dir / "short_side_under_20px.csv", index=False)
print(f"[짧은 변 20px 미만]: {len(short_small)}건")

# area 상·하위 1%
area_low  = valid["bbox_area"].quantile(0.01)
area_high = valid["bbox_area"].quantile(0.99)
area_extreme = valid[(valid["bbox_area"] < area_low) | (valid["bbox_area"] > area_high)]
area_extreme.to_csv(rev_dir / "area_extreme_1pct.csv", index=False)
print(f"[area 상·하위 1%] 하위 < {area_low:.0f}, 상위 > {area_high:.0f}: {len(area_extreme)}건")

# near-duplicate (완전 중복 제외하고 위치 유사한 것)
near_key = ["file_name", "bbox_x", "bbox_y"]
near_mask = df.duplicated(subset=near_key, keep=False) & ~df.duplicated(subset=dup_key, keep=False)
near_dup = df[near_mask]
near_dup.to_csv(rev_dir / "near_duplicate.csv", index=False)
print(f"[near-duplicate (위치 유사)]: {len(near_dup)}건")

print(f"\n점검 후보 CSV 저장: {rev_dir}")

[area 오차율]
  > 10%  (점검 후보)        : 191건
  > 100% (오류 후보)        : 54건
  > 500% (강한 오류 후보)   : 40건

[종횡비] 하위 1% < 0.406, 상위 1% > 1.859: 818건
[짧은 변 20px 미만]: 5건
[area 상·하위 1%] 하위 < 26000, 상위 > 213020: 815건
[near-duplicate (위치 유사)]: 4건

점검 후보 CSV 저장: D:\pill_detection\data\team_preprocessed\review_candidates


## Step 7. 최종 저장

- `clean_df.csv` — 전처리 완료 annotation
- `remove_list.json` — 제거 이미지 목록 (이유별)
- `summary.json` — 집계

In [9]:
# clean_df
clean_df = df[~df["file_name"].isin(all_remove)].copy()

print(f"원본  : {len(df):,} annotation / {df['file_name'].nunique():,} 이미지")
print(f"제거  : {len(df) - len(clean_df):,} annotation / {len(all_remove):,} 이미지")
print(f"잔존  : {len(clean_df):,} annotation / {clean_df['file_name'].nunique():,} 이미지")

clean_df.to_csv(Path(OUTPUT_DIR, "clean_df.csv"), index=False)
print(f"\nclean_df.csv 저장")

# remove_list.json
remove_list_json = {reason: sorted(list(imgs)) for reason, imgs in remove_images.items()}
with open(Path(OUTPUT_DIR, "remove_list.json"), "w", encoding="utf-8") as f:
    json.dump(remove_list_json, f, ensure_ascii=False, indent=2)
print("remove_list.json 저장")

# summary.json
summary = {
    "input": {
        "annotations": len(df),
        "images": int(df["file_name"].nunique()),
        "classes": int(df["category_id_merged"].nunique()),
    },
    "removed_by_reason": {r: len(imgs) for r, imgs in remove_images.items()},
    "removed_total_images": len(all_remove),
    "output": {
        "annotations": len(clean_df),
        "images": int(clean_df["file_name"].nunique()),
        "classes": int(clean_df["category_id_merged"].nunique()),
    },
    "review_candidates": {
        "area_error_over_10":  len(area_err_10),
        "area_error_over_100": len(area_err_100),
        "area_error_over_500": len(area_err_500),
        "aspect_ratio_extreme": len(aspect_extreme),
        "short_side_under_20px": len(short_small),
        "area_extreme_1pct": len(area_extreme),
        "near_duplicate": len(near_dup),
    }
}
with open(Path(OUTPUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n=== 요약 ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))

원본  : 40,844 annotation / 10,697 이미지
제거  : 980 annotation / 305 이미지
잔존  : 39,864 annotation / 10,392 이미지

clean_df.csv 저장
remove_list.json 저장

=== 요약 ===
{
  "input": {
    "annotations": 40844,
    "images": 10697,
    "classes": 118
  },
  "removed_by_reason": {
    "invalid_bbox": 2,
    "out_of_bounds": 1,
    "duplicate_bbox": 71,
    "label_mismatch": 232
  },
  "removed_total_images": 305,
  "output": {
    "annotations": 39864,
    "images": 10392,
    "classes": 118
  },
  "review_candidates": {
    "area_error_over_10": 191,
    "area_error_over_100": 54,
    "area_error_over_500": 40,
    "aspect_ratio_extreme": 818,
    "short_side_under_20px": 5,
    "area_extreme_1pct": 815,
    "near_duplicate": 4
  }
}


## 팀원 간 검증

`summary.json`을 팀에 공유해서 팀원 간 수치 일치 확인.

**팀원 전원이 동일한 수치가 나와야 함.**

In [10]:
print("=" * 50)
print("검증용 요약")
print("=" * 50)
print(f"제거 이미지 (union): {len(all_remove):,}")
for reason, imgs in remove_images.items():
    print(f"  {reason:20s}: {len(imgs):>5}")
print(f"\n최종 clean: {len(clean_df):,} annotation / {clean_df['file_name'].nunique():,} 이미지")
print("\n이 수치를 팀에 공유하세요.")

검증용 요약
제거 이미지 (union): 305
  invalid_bbox        :     2
  out_of_bounds       :     1
  duplicate_bbox      :    71
  label_mismatch      :   232

최종 clean: 39,864 annotation / 10,392 이미지

이 수치를 팀에 공유하세요.
